# Column Report Generator

This notebook generates a comprehensive markdown report for one or more columns.
Define the columns to analyze in the configuration cell below.

In [17]:
# =============================================================================
# CONFIGURATION - Define columns to analyze
# =============================================================================

# Table name (typed layer)
TABLE_NAME = "kontobuchungen"

# Columns to include in report (use None for all columns)
COLUMNS = [
    "ID des Rechnungslegungsbereichs",
    "Kontonummer des Kontos",
]
# COLUMNS = None  # Uncomment for all columns

# Output directory
OUTPUT_DIR = "./pipeline_output"

In [18]:
# =============================================================================
# Setup: Database connections and helpers
# =============================================================================

import json
import sqlite3
from datetime import datetime
from pathlib import Path

import pandas as pd

output_dir = Path(OUTPUT_DIR)
sqlite_path = output_dir / "metadata.db"
conn = sqlite3.connect(sqlite_path)


def sql(query):
    """Execute SQL query and return DataFrame."""
    return pd.read_sql_query(query, conn)


def json_pretty(json_str):
    """Pretty format JSON string."""
    if not json_str:
        return None
    try:
        return json.loads(json_str)
    except json.JSONDecodeError, TypeError:
        return json_str


# Report data collector
report_data = {
    "columns": {},
    "generated_at": datetime.now().isoformat(),
    "table_name": TABLE_NAME,
}

print(f"Connected to: {sqlite_path}")
print(f"Analyzing table: {TABLE_NAME}")
print(f"Columns filter: {COLUMNS if COLUMNS else 'All columns'}")

Connected to: pipeline_output/metadata.db
Analyzing table: kontobuchungen
Columns filter: ['ID des Rechnungslegungsbereichs', 'Kontonummer des Kontos']


In [19]:
# =============================================================================
# Step 1: Get base column information
# =============================================================================

# Build column filter clause
if COLUMNS:
    columns_list = ", ".join([f"'{c}'" for c in COLUMNS])
    column_filter = f"AND c.column_name IN ({columns_list})"
else:
    column_filter = ""

base_query = f"""
SELECT
    c.column_id,
    c.column_name,
    c.column_position,
    t.table_name,
    t.layer
FROM columns c
JOIN tables t ON c.table_id = t.table_id
WHERE t.table_name = '{TABLE_NAME}'
  AND t.layer = 'typed'
  {column_filter}
ORDER BY c.column_position
"""

df_base = sql(base_query)
column_ids = df_base["column_id"].tolist()
column_names = df_base["column_name"].tolist()

print(f"Found {len(df_base)} columns to analyze:")
for _, row in df_base.iterrows():
    print(f"  - {row['column_name']} (pos: {row['column_position']})")
    report_data["columns"][row["column_name"]] = {
        "column_id": row["column_id"],
        "column_position": row["column_position"],
    }

Found 2 columns to analyze:
  - ID des Rechnungslegungsbereichs (pos: 0)
  - Kontonummer des Kontos (pos: 1)


In [20]:
# =============================================================================
# Step 2: Type Decisions
# =============================================================================

column_ids_str = ", ".join([f"'{c}'" for c in column_ids])

type_query = f"""
SELECT
    c.column_name,
    td.decided_type,
    td.decision_source,
    td.decision_reason,
    tc.parse_success_rate,
    tc.detected_pattern,
    tc.detected_unit,
    tc.unit_confidence
FROM type_decisions td
JOIN columns c ON td.column_id = c.column_id
LEFT JOIN type_candidates tc ON td.column_id = tc.column_id AND td.decided_type = tc.data_type
WHERE td.column_id IN ({column_ids_str})
"""

df_types = sql(type_query)
print("=== TYPE DECISIONS ===")
display(df_types)

# Store in report
for _, row in df_types.iterrows():
    col_name = row["column_name"]
    if col_name in report_data["columns"]:
        report_data["columns"][col_name]["typing"] = {
            "decided_type": row["decided_type"],
            "decision_source": row["decision_source"],
            "decision_reason": row["decision_reason"],
            "parse_success_rate": row["parse_success_rate"],
            "detected_pattern": row["detected_pattern"],
            "detected_unit": row["detected_unit"],
            "unit_confidence": row["unit_confidence"],
        }

=== TYPE DECISIONS ===


,column_name,decided_type,decision_source,decision_reason,parse_success_rate,detected_pattern,detected_unit,unit_confidence
0,ID des Rechnungslegungsbereichs,BIGINT,automatic,Best candidate with confidence 1.00 (pattern: ...,1.0,integer,None,None
1,Kontonummer des Kontos,BIGINT,automatic,Best candidate with confidence 1.00 (pattern: ...,1.0,integer,None,None


In [21]:
# =============================================================================
# Step 3: Statistical Profiles
# =============================================================================

stats_query = f"""
SELECT
    c.column_name,
    sp.total_count,
    sp.null_count,
    sp.null_ratio,
    sp.distinct_count,
    sp.cardinality_ratio,
    sp.profile_data
FROM statistical_profiles sp
JOIN columns c ON sp.column_id = c.column_id
WHERE sp.column_id IN ({column_ids_str})
"""

df_stats = sql(stats_query)
print("=== STATISTICAL PROFILES ===")
display(
    df_stats[
        [
            "column_name",
            "total_count",
            "null_count",
            "null_ratio",
            "distinct_count",
            "cardinality_ratio",
        ]
    ]
)

# Store in report
for _, row in df_stats.iterrows():
    col_name = row["column_name"]
    if col_name in report_data["columns"]:
        profile_data = json_pretty(row["profile_data"])
        report_data["columns"][col_name]["statistics"] = {
            "total_count": row["total_count"],
            "null_count": row["null_count"],
            "null_ratio": row["null_ratio"],
            "distinct_count": row["distinct_count"],
            "cardinality_ratio": row["cardinality_ratio"],
            "profile_data": profile_data,
        }

=== STATISTICAL PROFILES ===


,column_name,total_count,null_count,null_ratio,distinct_count,cardinality_ratio
0,ID des Rechnungslegungsbereichs,3292,0,0.0,3,0.000911
1,Kontonummer des Kontos,3292,0,0.0,363,0.110267


In [22]:
# =============================================================================
# Step 4: Statistical Quality Metrics (Outliers, Benford)
# =============================================================================

quality_query = f"""
SELECT
    c.column_name,
    sqm.benford_compliant,
    sqm.has_outliers,
    sqm.iqr_outlier_ratio,
    sqm.isolation_forest_anomaly_ratio,
    sqm.quality_data
FROM statistical_quality_metrics sqm
JOIN columns c ON sqm.column_id = c.column_id
WHERE sqm.column_id IN ({column_ids_str})
"""

df_quality = sql(quality_query)
print("=== STATISTICAL QUALITY METRICS ===")
display(
    df_quality[
        [
            "column_name",
            "benford_compliant",
            "has_outliers",
            "iqr_outlier_ratio",
            "isolation_forest_anomaly_ratio",
        ]
    ]
)

# Store in report
for _, row in df_quality.iterrows():
    col_name = row["column_name"]
    if col_name in report_data["columns"]:
        quality_data = json_pretty(row["quality_data"])
        report_data["columns"][col_name]["quality"] = {
            "benford_compliant": bool(row["benford_compliant"])
            if row["benford_compliant"] is not None
            else None,
            "has_outliers": bool(row["has_outliers"]) if row["has_outliers"] is not None else None,
            "iqr_outlier_ratio": row["iqr_outlier_ratio"],
            "isolation_forest_anomaly_ratio": row["isolation_forest_anomaly_ratio"],
            "quality_data": quality_data,
        }

=== STATISTICAL QUALITY METRICS ===


,column_name,benford_compliant,has_outliers,iqr_outlier_ratio,isolation_forest_anomaly_ratio
0,ID des Rechnungslegungsbereichs,0,1,0.080194,0.040705
1,Kontonummer des Kontos,0,1,0.166160,0.049210


In [23]:
# =============================================================================
# Step 5: Semantic Annotations
# =============================================================================

semantic_query = f"""
SELECT
    c.column_name,
    sa.semantic_role,
    sa.entity_type,
    sa.business_name,
    sa.business_description,
    sa.business_concept,
    sa.confidence,
    sa.annotation_source
FROM semantic_annotations sa
JOIN columns c ON sa.column_id = c.column_id
WHERE sa.column_id IN ({column_ids_str})
"""

df_semantic = sql(semantic_query)
print("=== SEMANTIC ANNOTATIONS ===")
display(df_semantic)

# Store in report
for _, row in df_semantic.iterrows():
    col_name = row["column_name"]
    if col_name in report_data["columns"]:
        report_data["columns"][col_name]["semantic"] = {
            "semantic_role": row["semantic_role"],
            "entity_type": row["entity_type"],
            "business_name": row["business_name"],
            "business_description": row["business_description"],
            "business_concept": row["business_concept"],
            "confidence": row["confidence"],
            "annotation_source": row["annotation_source"],
        }

=== SEMANTIC ANNOTATIONS ===


,column_name,semantic_role,entity_type,business_name,business_description,business_concept,confidence,annotation_source
0,ID des Rechnungslegungsbereichs,dimension,accounting_area_id,Accounting Area ID,Identifier for the accounting area or ledger s...,entity,0.90,llm
1,Kontonummer des Kontos,dimension,account_number,Account Number,Chart of accounts number identifying the gener...,account,0.95,llm


In [24]:
# =============================================================================
# Step 6: Entropy Objects (all dimensions)
# =============================================================================

column_names_str = ", ".join([f"'{c}'" for c in column_names])

entropy_query = f"""
SELECT
    REPLACE(REPLACE(eo.target, 'column:{TABLE_NAME}.', ''), 'column:', '') AS column_name,
    eo.layer,
    eo.dimension,
    eo.sub_dimension,
    eo.score,
    eo.confidence,
    eo.evidence,
    eo.resolution_options
FROM entropy_objects eo
WHERE REPLACE(REPLACE(eo.target, 'column:{TABLE_NAME}.', ''), 'column:', '') IN ({column_names_str})
ORDER BY eo.target, eo.layer, eo.dimension
"""

df_entropy = sql(entropy_query)
print("=== ENTROPY OBJECTS ===")
display(df_entropy[["column_name", "layer", "dimension", "sub_dimension", "score", "confidence"]])

# Store in report (grouped by column)
for _, row in df_entropy.iterrows():
    col_name = row["column_name"]
    if col_name in report_data["columns"]:
        if "entropy" not in report_data["columns"][col_name]:
            report_data["columns"][col_name]["entropy"] = {}

        dim_key = f"{row['layer']}.{row['dimension']}"
        if row["sub_dimension"]:
            dim_key += f".{row['sub_dimension']}"

        report_data["columns"][col_name]["entropy"][dim_key] = {
            "score": row["score"],
            "confidence": row["confidence"],
            "evidence": json_pretty(row["evidence"]),
            "resolution_options": json_pretty(row["resolution_options"]),
        }

=== ENTROPY OBJECTS ===


,column_name,layer,dimension,sub_dimension,score,confidence
0,ID des Rechnungslegungsbereichs,computational,derived_values,formula_match,0.000000,1.0
1,ID des Rechnungslegungsbereichs,semantic,business_meaning,naming_clarity,0.200000,1.0
2,ID des Rechnungslegungsbereichs,semantic,dimensional,column_quality,0.050000,0.9
3,ID des Rechnungslegungsbereichs,structural,types,type_fidelity,0.000000,1.0
4,ID des Rechnungslegungsbereichs,value,nulls,null_ratio,0.000000,1.0
5,ID des Rechnungslegungsbereichs,value,outliers,outlier_rate,0.801944,1.0
6,Kontonummer des Kontos,semantic,business_meaning,naming_clarity,0.200000,1.0
7,Kontonummer des Kontos,semantic,dimensional,column_quality,0.050000,0.9
8,Kontonummer des Kontos,structural,types,type_fidelity,0.000000,1.0
9,Kontonummer des Kontos,value,nulls,null_ratio,0.000000,1.0


In [25]:
# =============================================================================
# Step 7: Entropy Interpretations (LLM-generated)
# =============================================================================

interp_query = f"""
SELECT
    ei.column_name,
    ei.composite_score,
    ei.readiness,
    ei.explanation,
    ei.assumptions_json,
    ei.resolution_actions_json
FROM entropy_interpretations ei
WHERE ei.table_name = '{TABLE_NAME}'
  AND ei.column_name IN ({column_names_str})
"""

df_interp = sql(interp_query)
print("=== ENTROPY INTERPRETATIONS ===")
display(df_interp[["column_name", "composite_score", "readiness"]])

# Store in report
for _, row in df_interp.iterrows():
    col_name = row["column_name"]
    if col_name in report_data["columns"]:
        report_data["columns"][col_name]["interpretation"] = {
            "composite_score": row["composite_score"],
            "readiness": row["readiness"],
            "explanation": row["explanation"],
            "assumptions": json_pretty(row["assumptions_json"]),
            "resolution_actions": json_pretty(row["resolution_actions_json"]),
        }

=== ENTROPY INTERPRETATIONS ===


,column_name,composite_score,readiness
0,ID des Rechnungslegungsbereichs,0.180292,ready
1,Kontonummer des Kontos,0.210000,ready


In [26]:
# =============================================================================
# Step 8: Relationships (if any)
# =============================================================================

rel_query = f"""
SELECT
    c.column_name AS from_column,
    r.relationship_type,
    r.left_cardinality,
    r.right_cardinality,
    r.is_confirmed,
    tf.column_name AS to_column,
    tt.table_name AS to_table
FROM relationships r
JOIN columns c ON r.left_column_id = c.column_id
JOIN columns tf ON r.right_column_id = tf.column_id
JOIN tables tt ON tf.table_id = tt.table_id
WHERE r.left_column_id IN ({column_ids_str})
   OR r.right_column_id IN ({column_ids_str})
"""

df_rel = sql(rel_query)
if len(df_rel) > 0:
    print("=== RELATIONSHIPS ===")
    display(df_rel)

    for _, row in df_rel.iterrows():
        col_name = row["from_column"]
        if col_name in report_data["columns"]:
            if "relationships" not in report_data["columns"][col_name]:
                report_data["columns"][col_name]["relationships"] = []
            report_data["columns"][col_name]["relationships"].append(
                {
                    "type": row["relationship_type"],
                    "to_table": row["to_table"],
                    "to_column": row["to_column"],
                    "cardinality": f"{row['left_cardinality']}:{row['right_cardinality']}",
                    "confirmed": bool(row["is_confirmed"]),
                }
            )
else:
    print("No relationships found for selected columns.")

DatabaseError: Execution failed on sql '
SELECT
    c.column_name AS from_column,
    r.relationship_type,
    r.left_cardinality,
    r.right_cardinality,
    r.is_confirmed,
    tf.column_name AS to_column,
    tt.table_name AS to_table
FROM relationships r
JOIN columns c ON r.left_column_id = c.column_id
JOIN columns tf ON r.right_column_id = tf.column_id
JOIN tables tt ON tf.table_id = tt.table_id
WHERE r.left_column_id IN ('a409bf4a-79bf-4132-b7d7-c6e0a5ebf26e', 'cf13ab15-7f7b-4f9c-8e47-ac427022d9ef')
   OR r.right_column_id IN ('a409bf4a-79bf-4132-b7d7-c6e0a5ebf26e', 'cf13ab15-7f7b-4f9c-8e47-ac427022d9ef')
': no such column: r.left_cardinality

In [27]:
# =============================================================================
# Step 9: Sample Data (from DuckDB)
# =============================================================================

import duckdb

duckdb_path = output_dir / "data.duckdb"
duck_conn = duckdb.connect(str(duckdb_path), read_only=True)

# Get sample values for each column
for col_name in column_names:
    try:
        sample_query = f"""
        SELECT "{col_name}" as value, COUNT(*) as count
        FROM typed_{TABLE_NAME}
        WHERE "{col_name}" IS NOT NULL
        GROUP BY "{col_name}"
        ORDER BY count DESC
        LIMIT 5
        """
        df_sample = duck_conn.execute(sample_query).df()

        if col_name in report_data["columns"]:
            report_data["columns"][col_name]["sample_values"] = [
                {"value": str(row["value"]), "count": int(row["count"])}
                for _, row in df_sample.iterrows()
            ]
    except Exception as e:
        print(f"Could not get samples for {col_name}: {e}")

duck_conn.close()
print(f"Collected sample values for {len(column_names)} columns.")

Collected sample values for 2 columns.


In [28]:
# =============================================================================
# GENERATE MARKDOWN REPORT
# =============================================================================


def generate_markdown_report(data: dict) -> str:
    """Generate a comprehensive markdown report from collected data."""
    lines = []

    # Header
    lines.append(f"# Column Report: {data['table_name']}")
    lines.append("")
    lines.append(f"**Generated:** {data['generated_at']}")
    lines.append(f"**Columns analyzed:** {len(data['columns'])}")
    lines.append("")
    lines.append("---")
    lines.append("")

    # Table of Contents
    lines.append("## Table of Contents")
    lines.append("")
    for col_name in data["columns"].keys():
        anchor = col_name.lower().replace(" ", "-").replace(".", "").replace(":", "")
        lines.append(f"- [{col_name}](#{anchor})")
    lines.append("")
    lines.append("---")
    lines.append("")

    # Each column
    for col_name, col_data in data["columns"].items():
        lines.append(f"## {col_name}")
        lines.append("")

        # Summary box
        typing = col_data.get("typing", {})
        semantic = col_data.get("semantic", {})
        interp = col_data.get("interpretation", {})

        lines.append("| Property | Value |")
        lines.append("|----------|-------|")
        lines.append(f"| **Type** | `{typing.get('decided_type', 'N/A')}` |")
        lines.append(f"| **Semantic Role** | {semantic.get('semantic_role', 'N/A')} |")
        lines.append(f"| **Business Name** | {semantic.get('business_name', 'N/A')} |")
        lines.append(f"| **Composite Entropy** | {interp.get('composite_score', 'N/A')} |")
        lines.append(f"| **Readiness** | {interp.get('readiness', 'N/A')} |")
        lines.append("")

        # Typing details
        if typing:
            lines.append("### Typing")
            lines.append("")
            lines.append(f"- **Decided Type:** `{typing.get('decided_type')}`")
            lines.append(f"- **Decision Source:** {typing.get('decision_source')}")
            lines.append(f"- **Parse Success Rate:** {typing.get('parse_success_rate')}")
            if typing.get("detected_pattern"):
                lines.append(f"- **Detected Pattern:** {typing.get('detected_pattern')}")
            if typing.get("detected_unit"):
                lines.append(
                    f"- **Detected Unit:** {typing.get('detected_unit')} (confidence: {typing.get('unit_confidence')})"
                )
            lines.append("")

        # Statistics
        stats = col_data.get("statistics", {})
        if stats:
            lines.append("### Statistics")
            lines.append("")
            lines.append(f"- **Total Count:** {stats.get('total_count'):,}")
            lines.append(
                f"- **Null Count:** {stats.get('null_count'):,} ({stats.get('null_ratio', 0) * 100:.1f}%)"
            )
            lines.append(f"- **Distinct Count:** {stats.get('distinct_count'):,}")
            lines.append(f"- **Cardinality Ratio:** {stats.get('cardinality_ratio', 0):.4f}")

            # Numeric stats if available
            profile = stats.get("profile_data", {})
            if profile and isinstance(profile, dict):
                numeric = profile.get("numeric_stats")
                if numeric:
                    lines.append("")
                    lines.append("**Numeric Statistics:**")
                    lines.append(f"- Min: {numeric.get('min_value')}")
                    lines.append(f"- Max: {numeric.get('max_value')}")
                    lines.append(f"- Mean: {numeric.get('mean')}")
                    lines.append(f"- Std Dev: {numeric.get('stddev')}")
            lines.append("")

        # Quality metrics
        quality = col_data.get("quality", {})
        if quality:
            lines.append("### Quality Metrics")
            lines.append("")
            if quality.get("benford_compliant") is not None:
                status = "Compliant" if quality.get("benford_compliant") else "Non-compliant"
                lines.append(f"- **Benford's Law:** {status}")
            if quality.get("has_outliers") is not None:
                lines.append(f"- **Has Outliers:** {quality.get('has_outliers')}")
            if quality.get("iqr_outlier_ratio") is not None:
                lines.append(
                    f"- **IQR Outlier Ratio:** {quality.get('iqr_outlier_ratio', 0) * 100:.2f}%"
                )
            if quality.get("isolation_forest_anomaly_ratio") is not None:
                lines.append(
                    f"- **Isolation Forest Anomaly Ratio:** {quality.get('isolation_forest_anomaly_ratio', 0) * 100:.2f}%"
                )
            lines.append("")

        # Semantic
        if semantic:
            lines.append("### Semantic Annotation")
            lines.append("")
            lines.append(f"- **Role:** {semantic.get('semantic_role')}")
            lines.append(f"- **Entity Type:** {semantic.get('entity_type')}")
            lines.append(f"- **Business Concept:** {semantic.get('business_concept')}")
            lines.append(f"- **Confidence:** {semantic.get('confidence')}")
            if semantic.get("business_description"):
                lines.append("")
                lines.append(f"**Description:** {semantic.get('business_description')}")
            lines.append("")

        # Entropy scores
        entropy = col_data.get("entropy", {})
        if entropy:
            lines.append("### Entropy Scores")
            lines.append("")
            lines.append("| Dimension | Score | Confidence |")
            lines.append("|-----------|-------|------------|")
            for dim, ent_data in sorted(entropy.items()):
                score = ent_data.get("score", 0)
                conf = ent_data.get("confidence", "N/A")
                # Color coding hint
                if score <= 0.2:
                    indicator = "Low"
                elif score <= 0.5:
                    indicator = "Medium"
                else:
                    indicator = "High"
                lines.append(f"| {dim} | {score:.3f} ({indicator}) | {conf} |")
            lines.append("")

            # Resolution options for high entropy dimensions
            high_entropy_dims = [(d, e) for d, e in entropy.items() if e.get("score", 0) > 0.3]
            if high_entropy_dims:
                lines.append("**Resolution Options (for high entropy dimensions):**")
                lines.append("")
                for dim, ent_data in high_entropy_dims:
                    res_opts = ent_data.get("resolution_options", [])
                    if res_opts and isinstance(res_opts, list):
                        lines.append(f"*{dim}:*")
                        for opt in res_opts[:3]:  # Top 3
                            if isinstance(opt, dict):
                                action = opt.get("action", "N/A")
                                desc = opt.get("description", "")
                                lines.append(f"- `{action}`: {desc}")
                        lines.append("")

        # Interpretation
        if interp:
            lines.append("### Interpretation")
            lines.append("")
            if interp.get("explanation"):
                lines.append(f"> {interp.get('explanation')}")
                lines.append("")

            # Assumptions - formatted nicely
            assumptions = interp.get("assumptions", [])
            if assumptions and isinstance(assumptions, list) and len(assumptions) > 0:
                lines.append("**Assumptions:**")
                lines.append("")
                for i, a in enumerate(assumptions, 1):
                    if isinstance(a, dict):
                        assumption_text = a.get("assumption_text", a.get("assumption", str(a)))
                        dimension = a.get("dimension", "")
                        confidence = a.get("confidence", "")
                        impact = a.get("impact", "")
                        basis = a.get("basis", "")

                        lines.append(
                            f"**{i}. {dimension}** (confidence: {confidence}, basis: {basis})"
                        )
                        lines.append(f"   - {assumption_text}")
                        if impact:
                            lines.append(f"   - *Impact if wrong:* {impact}")
                        lines.append("")
                    else:
                        lines.append(f"- {a}")
                lines.append("")

            # Resolution actions - formatted as table with all fields
            actions = interp.get("resolution_actions", [])
            if actions and isinstance(actions, list) and len(actions) > 0:
                lines.append("**Recommended Actions:**")
                lines.append("")
                lines.append("| Priority | Action | Description | Effort | Expected Impact |")
                lines.append("|----------|--------|-------------|--------|-----------------|")
                for a in actions:
                    if isinstance(a, dict):
                        priority = a.get("priority", "?")
                        action = a.get("action", "N/A")
                        desc = a.get("description", "")
                        effort = a.get("effort", "?")
                        expected_impact = a.get("expected_impact", "")
                        # Truncate long descriptions for table
                        desc_short = desc[:60] + "..." if len(desc) > 60 else desc
                        impact_short = (
                            expected_impact[:60] + "..."
                            if len(expected_impact) > 60
                            else expected_impact
                        )
                        lines.append(
                            f"| {priority} | `{action}` | {desc_short} | {effort} | {impact_short} |"
                        )
                    else:
                        lines.append(f"| ? | {a} | - | ? | - |")
                lines.append("")

                # Also add detailed view for each action
                lines.append("<details>")
                lines.append("<summary>Action Details (click to expand)</summary>")
                lines.append("")
                for i, a in enumerate(actions, 1):
                    if isinstance(a, dict):
                        lines.append(f"**{i}. `{a.get('action', 'N/A')}`**")
                        lines.append(f"- **Priority:** {a.get('priority', '?')}")
                        lines.append(f"- **Effort:** {a.get('effort', '?')}")
                        lines.append(f"- **Description:** {a.get('description', '')}")
                        lines.append(f"- **Expected Impact:** {a.get('expected_impact', '')}")
                        params = a.get("parameters", {})
                        if params:
                            lines.append(f"- **Parameters:** `{params}`")
                        lines.append("")
                lines.append("</details>")
                lines.append("")

        # Relationships
        rels = col_data.get("relationships", [])
        if rels:
            lines.append("### Relationships")
            lines.append("")
            for rel in rels:
                confirmed = "confirmed" if rel.get("confirmed") else "unconfirmed"
                lines.append(
                    f"- **{rel.get('type')}** -> `{rel.get('to_table')}.{rel.get('to_column')}` ({rel.get('cardinality')}, {confirmed})"
                )
            lines.append("")

        # Sample values
        samples = col_data.get("sample_values", [])
        if samples:
            lines.append("### Sample Values (Top 5)")
            lines.append("")
            lines.append("| Value | Count |")
            lines.append("|-------|-------|")
            for s in samples:
                val = s.get("value", "")[:50]  # Truncate long values
                if len(s.get("value", "")) > 50:
                    val += "..."
                lines.append(f"| `{val}` | {s.get('count'):,} |")
            lines.append("")

        lines.append("---")
        lines.append("")

    return "\n".join(lines)


# Generate the report
markdown_report = generate_markdown_report(report_data)
print(f"Generated markdown report: {len(markdown_report)} characters")

Generated markdown report: 10352 characters


In [29]:
# =============================================================================
# Save and Display Report
# =============================================================================

# Save to file
report_filename = f"column_report_{TABLE_NAME}.md"
report_path = output_dir / report_filename
with open(report_path, "w") as f:
    f.write(markdown_report)
print(f"Report saved to: {report_path}")

# Also save JSON for programmatic access
json_path = output_dir / f"column_report_{TABLE_NAME}.json"
with open(json_path, "w") as f:
    json.dump(report_data, f, indent=2, default=str)
print(f"JSON data saved to: {json_path}")

Report saved to: pipeline_output/column_report_kontobuchungen.md
JSON data saved to: pipeline_output/column_report_kontobuchungen.json


In [30]:
# =============================================================================
# Display Report (Markdown)
# =============================================================================

from IPython.display import Markdown, display

display(Markdown(markdown_report))

# Column Report: kontobuchungen

**Generated:** 2026-02-09T18:23:44.072814
**Columns analyzed:** 2

---

## Table of Contents

- [ID des Rechnungslegungsbereichs](#id-des-rechnungslegungsbereichs)
- [Kontonummer des Kontos](#kontonummer-des-kontos)

---

## ID des Rechnungslegungsbereichs

| Property | Value |
|----------|-------|
| **Type** | `BIGINT` |
| **Semantic Role** | dimension |
| **Business Name** | Accounting Area ID |
| **Composite Entropy** | 0.18029161603888214 |
| **Readiness** | ready |

### Typing

- **Decided Type:** `BIGINT`
- **Decision Source:** automatic
- **Parse Success Rate:** 1.0
- **Detected Pattern:** integer

### Statistics

- **Total Count:** 3,292
- **Null Count:** 0 (0.0%)
- **Distinct Count:** 3
- **Cardinality Ratio:** 0.0009

**Numeric Statistics:**
- Min: 0.0
- Max: 2.0
- Mean: 0.12089914945321993
- Std Dev: 0.43330018987769436

### Quality Metrics

- **Benford's Law:** Non-compliant
- **Has Outliers:** True
- **IQR Outlier Ratio:** 8.02%
- **Isolation Forest Anomaly Ratio:** 4.07%

### Semantic Annotation

- **Role:** dimension
- **Entity Type:** accounting_area_id
- **Business Concept:** entity
- **Confidence:** 0.9

**Description:** Identifier for the accounting area or ledger segment, used to separate different accounting books or legal entities.

### Entropy Scores

| Dimension | Score | Confidence |
|-----------|-------|------------|
| computational.derived_values.formula_match | 0.000 (Low) | 1.0 |
| semantic.business_meaning.naming_clarity | 0.200 (Low) | 1.0 |
| semantic.dimensional.column_quality | 0.050 (Low) | 0.9 |
| structural.types.type_fidelity | 0.000 (Low) | 1.0 |
| value.nulls.null_ratio | 0.000 (Low) | 1.0 |
| value.outliers.outlier_rate | 0.802 (High) | 1.0 |

**Resolution Options (for high entropy dimensions):**

*value.outliers.outlier_rate:*
- `winsorize`: Cap extreme values at specified percentiles
- `exclude_outliers`: Exclude IQR-based outliers from aggregations
- `investigate_outliers`: Manual review of outlier values for data quality issues

### Interpretation

> This accounting area identifier has excellent structural quality (BIGINT type) but shows a very high outlier rate (0.802), suggesting the distribution of accounting area IDs is highly skewed or contains unusual values. The column name has minor clarity issues. The data is marked 'ready' with a low composite score of 0.18, indicating it's usable but requires attention to the outlier pattern.

**Assumptions:**

**1. value.outliers.outlier_rate** (confidence: medium, basis: inferred)
   - The kontobuchungen.ID des Rechnungslegungsbereichs column contains legitimate accounting area identifiers, and the 80.2% outlier rate reflects a valid business pattern where most transactions concentrate in a few primary accounting areas while others are rarely used.
   - *Impact if wrong:* If incorrect, queries filtering or aggregating by accounting area may produce misleading results, potentially excluding valid transactions or including erroneous data that should be cleaned.

**2. semantic.business_meaning.naming_clarity** (confidence: high, basis: inferred)
   - The column 'ID des Rechnungslegungsbereichs' in kontobuchungen refers to a standard accounting area/ledger identifier used consistently across the organization, with values mapping to a known reference table.
   - *Impact if wrong:* Without clear documentation, analysts may misinterpret which accounting areas the IDs represent, leading to incorrect financial reporting or consolidation.


**Recommended Actions:**

| Priority | Action | Description | Effort | Expected Impact |
|----------|--------|-------------|--------|-----------------|
| high | `investigate_outlier_pattern` | Analyze the distribution of accounting area IDs to determine... | medium | Reduces value.outliers.outlier_rate uncertainty and validate... |
| medium | `add_reference_documentation` | Create or link to a reference table documenting valid accoun... | low | Reduces semantic.business_meaning.naming_clarity from 0.2 to... |

<details>
<summary>Action Details (click to expand)</summary>

**1. `investigate_outlier_pattern`**
- **Priority:** high
- **Effort:** medium
- **Description:** Analyze the distribution of accounting area IDs to determine if the 80.2% outlier rate represents valid business logic (e.g., most transactions in 1-2 main ledgers) or data quality issues requiring cleanup.
- **Expected Impact:** Reduces value.outliers.outlier_rate uncertainty and validates whether the skewed distribution is expected or indicates data problems.

**2. `add_reference_documentation`**
- **Priority:** medium
- **Effort:** low
- **Description:** Create or link to a reference table documenting valid accounting area IDs and their business meanings, improving semantic clarity.
- **Expected Impact:** Reduces semantic.business_meaning.naming_clarity from 0.2 to near 0.0 by providing clear mapping of IDs to business entities.

</details>

### Sample Values (Top 5)

| Value | Count |
|-------|-------|
| `0` | 3,028 |
| `2` | 134 |
| `1` | 130 |

---

## Kontonummer des Kontos

| Property | Value |
|----------|-------|
| **Type** | `BIGINT` |
| **Semantic Role** | dimension |
| **Business Name** | Account Number |
| **Composite Entropy** | 0.21 |
| **Readiness** | ready |

### Typing

- **Decided Type:** `BIGINT`
- **Decision Source:** automatic
- **Parse Success Rate:** 1.0
- **Detected Pattern:** integer

### Statistics

- **Total Count:** 3,292
- **Null Count:** 0 (0.0%)
- **Distinct Count:** 363
- **Cardinality Ratio:** 0.1103

**Numeric Statistics:**
- Min: 135.0
- Max: 91500.0
- Mean: 13815.852065613608
- Std Dev: 23887.17848952669

### Quality Metrics

- **Benford's Law:** Non-compliant
- **Has Outliers:** True
- **IQR Outlier Ratio:** 16.62%
- **Isolation Forest Anomaly Ratio:** 4.92%

### Semantic Annotation

- **Role:** dimension
- **Entity Type:** account_number
- **Business Concept:** account
- **Confidence:** 0.95

**Description:** Chart of accounts number identifying the general ledger account being debited or credited in this posting.

### Entropy Scores

| Dimension | Score | Confidence |
|-----------|-------|------------|
| semantic.business_meaning.naming_clarity | 0.200 (Low) | 1.0 |
| semantic.dimensional.column_quality | 0.050 (Low) | 0.9 |
| structural.types.type_fidelity | 0.000 (Low) | 1.0 |
| value.nulls.null_ratio | 0.000 (Low) | 1.0 |
| value.outliers.outlier_rate | 1.000 (High) | 1.0 |

**Resolution Options (for high entropy dimensions):**

*value.outliers.outlier_rate:*
- `winsorize`: Cap extreme values at specified percentiles
- `exclude_outliers`: Exclude IQR-based outliers from aggregations
- `investigate_outliers`: Manual review of outlier values for data quality issues

### Interpretation

> This general ledger account number column has a composite score of 0.21 and is 'ready'. The main concern is a perfect 1.0 outlier rate, meaning all account numbers are flagged as outliers. This is likely a false positive from outlier detection not being calibrated for account number ranges. No nulls (0.0) indicates complete data. The BIGINT type is appropriate for numeric account codes.

**Assumptions:**

**1. value.outliers.outlier_rate** (confidence: high, basis: inferred)
   - The 100% outlier rate in kontobuchungen.Kontonummer des Kontos is a false positive caused by the chart of accounts numbering scheme (e.g., ranges like 1000-1999 for assets, 4000-4999 for revenue), and all values represent valid account numbers from the chart of accounts.
   - *Impact if wrong:* If some account numbers are actually invalid or not in the chart of accounts, financial reporting could include erroneous postings or misclassify transactions.

**2. semantic.business_meaning.naming_clarity** (confidence: high, basis: inferred)
   - The kontobuchungen.Kontonummer des Kontos column contains account numbers that map to a documented chart of accounts with account descriptions, categories, and financial statement line items.
   - *Impact if wrong:* Without a chart of accounts mapping, analysts cannot interpret what each account represents, making financial analysis impossible.


**Recommended Actions:**

| Priority | Action | Description | Effort | Expected Impact |
|----------|--------|-------------|--------|-----------------|
| high | `add_reference_documentation` | Link to the chart of accounts (Kontenplan) that defines vali... | low | Reduces semantic.business_meaning.naming_clarity from 0.2 to... |
| medium | `recalibrate_outlier_detection` | Disable or recalibrate outlier detection for account numbers... | low | Reduces value.outliers.outlier_rate from 1.0 to 0.0 or a mea... |
| medium | `add_referential_integrity` | Implement a foreign key or validation rule linking Kontonumm... | medium | Prevents invalid account numbers from being entered, improvi... |

<details>
<summary>Action Details (click to expand)</summary>

**1. `add_reference_documentation`**
- **Priority:** high
- **Effort:** low
- **Description:** Link to the chart of accounts (Kontenplan) that defines valid account numbers, their descriptions, account types, and financial statement classifications.
- **Expected Impact:** Reduces semantic.business_meaning.naming_clarity from 0.2 to near 0.0, enabling proper interpretation of account numbers and financial reporting.

**2. `recalibrate_outlier_detection`**
- **Priority:** medium
- **Effort:** low
- **Description:** Disable or recalibrate outlier detection for account numbers, as they follow a structured chart of accounts scheme rather than a statistical distribution. Consider validation against the valid account list instead.
- **Expected Impact:** Reduces value.outliers.outlier_rate from 1.0 to 0.0 or a meaningful level that identifies truly invalid account numbers.

**3. `add_referential_integrity`**
- **Priority:** medium
- **Effort:** medium
- **Description:** Implement a foreign key or validation rule linking Kontonummer des Kontos to a master chart of accounts table to ensure only valid account numbers are used.
- **Expected Impact:** Prevents invalid account numbers from being entered, improving structural integrity and reducing future value_entropy.

</details>

### Sample Values (Top 5)

| Value | Count |
|-------|-------|
| `1200` | 360 |
| `3300` | 238 |
| `1406` | 200 |
| `1800` | 194 |
| `1830` | 187 |

---


In [31]:
# Cleanup
conn.close()
print("Done!")

Done!
